# Toy Test: YOLOv8-pose Pruning End-to-End

This notebook runs the simple pruning script on the tiny `coco8-pose` dataset (8 images, ~1 min on a single GPU) and verifies that:

1. Training completes without errors
2. The saved `last.pt` contains the actual pruned tensors (smaller params)
3. The `pruned_model.onnx` export reflects the real pruned architecture
4. Validation on the loaded model matches the auto-logged metrics

**Use this notebook as a smoke test** before kicking off a long `coco-pose` run, to verify your environment is set up correctly. Once it passes, swap `coco8-pose.yaml` → `coco-pose.yaml`, bump `--epochs` to 50, and launch the real run from the command line.

## 1. Environment check

Verify the four pinned dependencies are importable. If any fails, run `pip install -r requirements.txt` and restart the kernel.

In [1]:
import torch
import ultralytics
import torch_pruning as tp
import fasterai

print(f'torch         {torch.__version__}')
print(f'ultralytics   {ultralytics.__version__}')
print(f'torch-pruning {tp.__version__ if hasattr(tp, "__version__") else "(no version attr)"}')
print(f'fasterai      {fasterai.__version__ if hasattr(fasterai, "__version__") else "(no version attr)"}')
print(f'CUDA available: {torch.cuda.is_available()}')

torch         2.9.1+cu128
ultralytics   8.3.162
torch-pruning (no version attr)
fasterai      0.3.0
CUDA available: True


## 2. Run the simple pruning script

Tiny config: 4 epochs on coco8-pose at imgsz=320, batch=2. The pruning callback fires schedule-driven prune events; with `--asymmetric --asymmetric-scale 0.5` we target ~6% sparsity (gentle for a toy run).

Wall time: ~30–60 seconds on a modern GPU.

In [7]:
import subprocess
import sys

cmd = [
    sys.executable, 'prune_yolov8_pose_simple.py',
    '--data',  'coco8-pose.yaml',
    '--epochs', '2',
    '--imgsz',  '320',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print('--- tail of stdout ---')
print('\n'.join(result.stdout.splitlines()[-30:]))
if result.returncode != 0:
    print('--- stderr ---'); print(result.stderr[-2000:])
    raise RuntimeError(f'script exited with code {result.returncode}')

Running: /home/nathan/miniconda3/envs/dev/bin/python prune_yolov8_pose_simple.py --data coco8-pose.yaml --epochs 2 --imgsz 320
--- tail of stdout ---

[CB epoch 0 (pct_train=0.500)] Pruning step 1/2
   → 0.98 GMACs, 2.75M params (sparsity 16.6%, MACs −15.5%)

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
                   all          4         14      0.132      0.357      0.337      0.182      0.161      0.286      0.252     0.0879

[Epoch 1/2] sparsity=16.6%  MACs−15.5%  prunes=1/2  mAP50-95_P=0.0879  mAP50_P=0.2518  mAP50-95_B=0.1815  mAP50_B=0.3369

[CB epoch 1 (pct_train=1.000)] Pruning step 2/2
   → 0.92 GMACs, 2.57M params (sparsity 21.9%, MACs −20.1%)

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
                   all          4         14      0.103      0.214     0.0715     0.0228      0.069      0.143     0.0874     0.0206

[Epoch 2/2] sparsity=21.9%  MACs−20.1%  p

## 3. Inspect the auto-logged experiment

The script appends a line to `experiments.jsonl`. Read the last entry and check final params/mAP.

In [10]:
last_run

{'timestamp': '2026-05-18T12:06:15',
 'label': '2step_2ep_r0.12',
 'command': 'python prune_yolov8_pose.py --data coco8-pose.yaml --steps 2 --epochs 2 --imgsz 320 --batch 16 --ratio 0.12',
 'config': {'model': 'yolov8n-pose.pt',
  'data': 'coco8-pose.yaml',
  'imgsz': 320,
  'batch': 16,
  'steps': 2,
  'epochs': 2,
  'ratio': 0.12},
 'baseline': {'params_m': 3.295, 'macs_g': 1.155},
 'final': {'box_map50': 0.07364822361546501,
  'box_map5095': 0.022920843581705652,
  'pose_map50': 0.08740804597701149,
  'pose_map5095': 0.021095393457117598,
  'params_m': 2.57,
  'macs_g': 0.92,
  'param_reduction_pct': 22.03,
  'mac_reduction_pct': 20.35},
 'wall_time_s': 5.0,
 'notes': ''}

In [8]:
import json
from pathlib import Path

lines = Path('experiments.jsonl').read_text().strip().split('\n')
last_run = json.loads(lines[-1])

print(f'Label:           {last_run["label"]}')
print()
print(f'Baseline params: {last_run["baseline"]["params_m"]:.3f} M')
print(f'Final    params: {last_run["final"]["params_m"]:.3f} M')
print()
print(f'Param reduction: {last_run["final"]["param_reduction_pct"]:.2f} %')
print(f'MAC   reduction: {last_run["final"]["mac_reduction_pct"]:.2f} %')
print()
print(f'pose mAP50-95:   {last_run["final"]["pose_map5095"]:.4f}')
print(f'box  mAP50-95:   {last_run["final"]["box_map5095"]:.4f}')

Label:           2step_2ep_r0.12

Baseline params: 3.295 M
Final    params: 2.570 M

Param reduction: 22.03 %
MAC   reduction: 20.35 %

pose mAP50-95:   0.0211
box  mAP50-95:   0.0229


## 4. Verify `last.pt` IS the pruned model

The most recent training run lives at `runs/pose/train<N>/weights/last.pt` (latest N). Loading it requires importing `C2f_v2` first (the surgery class is needed by the unpickler), then `YOLO()` reconstructs the pruned architecture from the saved tensors.

**Cross-check**: the loaded model's parameter count should match `experiments.jsonl`'s `final.params_m` exactly.

In [9]:
import sys
sys.path.insert(0, '.')
from prune_yolov8_pose import C2f_v2  # required so pickle can deserialize
from ultralytics import YOLO

# Find the most recent train directory
train_dirs = sorted(Path('runs/pose').glob('train*'), key=lambda p: p.stat().st_mtime)
latest_pt = train_dirs[-1] / 'weights' / 'last.pt'
print(f'Loading: {latest_pt}')

yolo_loaded = YOLO(str(latest_pt))
loaded_params = sum(p.numel() for p in yolo_loaded.model.parameters()) / 1e6
print(f'Loaded model params: {loaded_params:.3f} M')
print(f'Match with log:      {"YES" if abs(loaded_params - last_run["final"]["params_m"]) < 0.01 else "NO"}')

# Show one representative pruned layer
model_7 = yolo_loaded.model.model[7]
if hasattr(model_7, 'conv'):
    print(f'\nmodel[7].conv weight shape: {tuple(model_7.conv.weight.shape)}')
    print('(baseline yolov8n-pose has shape (256, 128, 3, 3) — pruned should be smaller)')

Loading: runs/pose/train3/weights/last.pt
Loaded model params: 2.574 M
Match with log:      YES

model[7].conv weight shape: (225, 112, 3, 3)
(baseline yolov8n-pose has shape (256, 128, 3, 3) — pruned should be smaller)
